# Live MRAS — data assimilation, panel by panel

One run, six ngsplot panels, repainted as the solve progresses and stitched
into an animated HTML movie at the end:

| current datum $z$ | data error $u - z$ | ground truth $u$ at final time |
|---|---|---|
| estimate $\hat q$ | reconstruction error $\hat q - q$ | true parameter $q$ |

Everything is rendered by `ngsplot.Plot` — the same FEM picture NGSolve's
own viewers use — with each colorbar static, set by the extremes of its
category (state panels share one range, parameter panels share theirs,
error panels are symmetric about zero), so colours compare across time.
`live=True` streams the panels into the notebook: the figure opens at
t = 0, before the (sometimes long) data-generating forward run, then
refreshes as the solve advances — by default thinned to a fixed budget of
evenly spaced frames, so any horizon updates at a steady rate.  `dt` and
`tend` set the step size and horizon, `every` overrides the thinning, and
`fps`/`path` control the movie (default `results/<name>_noise<p>.html`).

Needs the `notebook` extra (`pip install -e ".[notebook]"`) and a Jupyter
front-end; everything else happens inside `mras/live.py`.


In [ ]:
name = "burgers_pipe"   # choose: burgers_pipe | darcy | fisher | potential | allen_cahn | arrhenius
noise_level = 0.0       # relative L2 noise of the data, as a fraction
                        # (0.0 = noise-free, 0.05 = noise level 5%)

In [ ]:
import matplotlib as mpl
from mras.live import run_live

mpl.rcParams["figure.dpi"] = 144  # crisp panels in the notebook

result = run_live(name, noise_level, live=True, dt=0.05, tend=200.0)


In [ ]:
# A headless kernel starts on the Agg backend: no figure ever reaches
# the notebook. The inline backend is what provides PNG figure output.
%matplotlib inline
import matplotlib.pyplot as plt

plt.plot(result.error_q)
plt.xlabel("assimilation step")
plt.ylabel(r"relative q error  ||qhat - q|| / ||q||")
plt.grid(True, alpha=0.3)


The Burgers demo identifies the wake field $q$ around the hole row from
aperiodic heater data: in the animation the lower-left panel lights up as
the incommensurate heaters drive the flow, the state tracks within O(dt),
and the reconstruction-error panel descends slowly but persistently —
noise-free, the recorded error $\|\hat q - q\| / \|q\|$ reaches about 0.22
by $t = 200$, 0.14 by $t = 400$ and 0.10 by $t = 800$; the horizon, not
the step size, is the lever for parameter accuracy (`dt` below 0.05 buys
`q` nothing). Seeded noise leaves the state tracking untouched but floors
the parameter error (noise level 5% bottoms near 0.5) — the
persistent-excitation caveat documented in `docs/DEVIATIONS.md` is the
honest statement of what the theory does not guarantee.
